## Instalacion de dependencias

In [1]:
!pip install chromadb langchain-chroma langchain-huggingface \
             langchain-groq sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

## Importacion de librerias

In [2]:
import os
import time
import pandas as pd
import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [6]:
import shutil, os, pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Limpiar disco
if os.path.exists("/content/chroma_reglamento"):
    shutil.rmtree("/content/chroma_reglamento")
print("Disco limpio.")

# Embedder
embedder = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1",
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True}
)

# Cargar chunks
df_chunks = pd.read_csv('chunks_reglamento.csv')
textos    = df_chunks['texto'].tolist()
metadatos = df_chunks[['chunk_id','fuente','tipo']].to_dict(orient='records')
metadatos = [{k: str(v) for k, v in m.items()} for m in metadatos]

# Insertar
print("Insertando chunks...")
vectorstore = Chroma.from_texts(
    texts=textos,
    embedding=embedder,
    metadatas=metadatos,
    persist_directory="/content/chroma_reglamento",
    collection_name="reglamento_universitario",
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"\n✅ ChromaDB lista: {vectorstore._collection.count()} chunks")

Disco limpio.


Insertando chunks...

✅ ChromaDB lista: 695 chunks


In [7]:
docs = vectorstore.similarity_search("inasistencias", k=3)
for i, doc in enumerate(docs):
    print(f"#{i+1}: {doc.page_content[:200]}")

#1: inmediato siguiente ni participar en lista alguna. 
 
ARTÍCULO 321°.- Los cargos de rector y vicerrector se ejercen a dedicación exclusiva y 
son incompatibles con el desempeño de cualquier otra funci
#2: certificados, títulos y otros conceptos; los recursos de balances de su propio presupuesto y 
las utilidades resultantes de la prestación de servicios o la venta de productos generados por 
su propia 
#3: ofrecidas por la UNALM. 
 
ARTÍCULO 104°.- Los dos primeros estudiantes, por orden de mérito de los centros 
educativos de nivel secundario del Perú, participan por las vacantes asignadas por el conse


## Carga de API KEY

Se carga la API Key de Groq desde Colab Secrets para no exponerla en el código.
La key se obtiene gratuitamente en https://console.groq.com

In [8]:
from google.colab import userdata

# Obtenida de https://console.groq.com
os.environ["GROQ_API_KEY"] = "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"

print("API Key cargada.")

API Key cargada.


## Prompt del sistema:
Este tiene el objetivo de:
- Instruir al LLM a responder solo con el contexto recuperado
- Pedir que cite el fragmento relevante
- Responder que no sabe si es que la informacion no esta presente

In [14]:
PROMPT_TEMPLATE = """Eres un asistente especializado en el reglamento universitario.
Responde SIEMPRE en español, sin excepción.
Responde la pregunta del estudiante ÚNICAMENTE basándote en el contexto proporcionado.
Si la respuesta no está en el contexto, responde exactamente:
"No encontré información sobre eso en el reglamento."

Contexto del reglamento:
{context}

Pregunta del estudiante: {question}

Respuesta clara y directa en español:"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print("Prompt configurado.")

Prompt configurado.


## Funcion RAG reutilizable

Se implementan dos funciones reutilizables:

**construir_rag(llm, vectorstore, top_k):**
Construye el pipeline completo:
Pregunta → Retriever (ChromaDB) → Contexto → Prompt → LLM → Respuesta

**preguntar(cadena, retriever, pregunta):**
Ejecuta una consulta, mide el tiempo de respuesta y muestra
los chunks recuperados como fuentes verificables.

In [15]:
def construir_rag(llm, vectorstore, top_k=5):
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k}
    )

    def formatear_contexto(docs):
        return "\n\n---\n\n".join([d.page_content for d in docs])

    cadena = (
        {"context": retriever | formatear_contexto,
         "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return cadena, retriever


def preguntar(cadena, retriever, pregunta, mostrar_fuentes=True):
    t0 = time.time()
    respuesta = cadena.invoke(pregunta)
    tiempo    = time.time() - t0

    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*60}")
    print(f"RESPUESTA ({tiempo:.1f}s):\n{respuesta}")

    if mostrar_fuentes:
        docs = retriever.invoke(pregunta)
        print(f"\nFUENTES RECUPERADAS (top-{len(docs)}):")
        for i, doc in enumerate(docs):
            print(f"\n  #{i+1} | {doc.page_content[:200]}...")

    print("\n" + "="*60 + "\n")
    return respuesta

## Preguntas finales:

Conjunto de 5 preguntas típicas de estudiantes universitarios usadas
para evaluar de forma consistente todos los experimentos de este notebook.

In [19]:
preguntas_prueba = [
    "¿Qué pasa si desapruebo el mismo curso tres veces?",       # ✅ funciona
    "¿Qué ocurre si desapruebo un curso por cuarta vez?",       # mismo artículo
    "¿Qué es la suspensión de estudios?",                       # mismo artículo
    "¿Cuáles son las modalidades de admisión?",                 # art 100° apareció
    "¿Qué requisitos necesito para obtener el grado de bachiller?",  # art 149° apareció
]

## Evaluacion para top-k: 3 vs 5 vs 7

In [20]:
llm_llama = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
resultados_k = []
for k in [3, 5, 7]:
    cadena, retriever = construir_rag(llm_llama, vectorstore, top_k=k)
    tiempos = []

    print(f"\n{'='*60}")
    print(f"Evaluando top-k = {k}")
    print(f"{'='*60}")

    for pregunta in preguntas_prueba[:3]:
        t0 = time.time()
        respuesta = cadena.invoke(pregunta)
        t  = time.time() - t0
        tiempos.append(t)

        print(f"\nPREGUNTA: {pregunta}")
        print(f"RESPUESTA: {respuesta[:300]}...")
        print(f"Tiempo: {t:.1f}s")

    resultados_k.append({
        "top_k":          k,
        "tiempo_medio_s": round(sum(tiempos)/len(tiempos), 2)
    })

print("\n=== Resumen tiempos por top-k ===")
print(pd.DataFrame(resultados_k).to_string(index=False))


Evaluando top-k = 3

PREGUNTA: ¿Qué pasa si desapruebo el mismo curso tres veces?
RESPUESTA: Según el artículo 139° del reglamento, si desapruebo el mismo curso o módulo de competencias por tercera vez, da lugar a la suspensión de mis estudios de la universidad por el período de un (1) semestre académico....
Tiempo: 0.6s

PREGUNTA: ¿Qué ocurre si desapruebo un curso por cuarta vez?
RESPUESTA: Si desapruebo un curso por cuarta vez, seré separado definitivamente de la UNALM....
Tiempo: 0.5s

PREGUNTA: ¿Qué es la suspensión de estudios?
RESPUESTA: La suspensión de estudios es la condición de los estudiantes que se matriculan en el semestre académico posterior al cumplimiento de su suspensión, según lo establecido en el artículo 413º inciso d) del reglamento....
Tiempo: 0.5s

Evaluando top-k = 5

PREGUNTA: ¿Qué pasa si desapruebo el mismo curso tres veces?
RESPUESTA: Según el artículo 139° del reglamento, si desapruebo el mismo curso tres veces, da lugar a la suspensión de mis estudios de

## Comparacion de LLama con Qwen

In [21]:
llm_llama = ChatGroq(model="llama-3.1-8b-instant",   temperature=0)
llm_qwen  = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

cadena_llama, retriever = construir_rag(llm_llama, vectorstore, top_k=5)
cadena_qwen,  _         = construir_rag(llm_qwen,  vectorstore, top_k=5)

comparaciones = []

for pregunta in preguntas_prueba:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*60}")

    t0 = time.time()
    resp_llama = cadena_llama.invoke(pregunta)
    t_llama    = time.time() - t0

    t0 = time.time()
    resp_qwen  = cadena_qwen.invoke(pregunta)
    t_qwen     = time.time() - t0

    print(f"\n[Llama 3 · {t_llama:.1f}s]\n{resp_llama}")
    print(f"\n[Qwen · {t_qwen:.1f}s]\n{resp_qwen}")

    comparaciones.append({
        "pregunta":    pregunta[:50],
        "t_llama_s":  round(t_llama, 2),
        "t_qwen_s":   round(t_qwen, 2),
        "long_llama":  len(resp_llama),
        "long_qwen":   len(resp_qwen)
    })

print("\n=== Tabla comparativa ===")
print(pd.DataFrame(comparaciones).to_string(index=False))


PREGUNTA: ¿Qué pasa si desapruebo el mismo curso tres veces?

[Llama 3 · 8.6s]
Según el artículo 139° del reglamento, si desapruebo el mismo curso tres veces, da lugar a la suspensión de mis estudios de la universidad por el período de un (1) semestre académico.

[Qwen · 1.2s]
<think>
Okay, the student is asking what happens if they fail the same course three times. Let me check the provided context.

Looking through the articles, Article 139 mentions that failing the same course or module three times leads to suspension for one semester. After that, they can only enroll in that specific course again and return regularly the following semester. If they fail it a fourth time, they're separated definitively from the university. 

So the answer should state that after three failures, there's a one-semester suspension, and they can only take that course again. I need to make sure I don't include other articles unless relevant. The other articles talk about different things like the univer

In [42]:
cadena_final, retriever_final = construir_rag(
    llm_llama, vectorstore, top_k=5
)
preguntar(cadena_final, retriever_final,
          "¿Cuáles son los requisitos para sustentar la tesis?",
          mostrar_fuentes=True)

PREGUNTA: ¿Cuáles son los requisitos para sustentar la tesis?
RESPUESTA (0.4s):
No encontré información sobre eso en el reglamento.

FUENTES RECUPERADAS (top-0):




'No encontré información sobre eso en el reglamento.'

In [23]:
from google.colab import files
import shutil

shutil.make_archive("chroma_reglamento", "zip", "/content/chroma_reglamento")
files.download("chroma_reglamento.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descargando chroma_reglamento.zip...
